# B2 参考答案 notebook
指数训练集构建 + 时序切分 + KS（填空已完成）。
注意：本题用于训练集质量与偏差检测，不是实盘交易策略。


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from scipy import stats
from sklearn.model_selection import TimeSeriesSplit

def load_and_preprocess_data(file_path):
    """加载并预处理指数数据"""
    try:
        df = pd.read_csv(file_path)

        # 将日期时间转换为datetime类型
        df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'])

        # M1提取日期
        df['date_only'] = df['datetime'].dt.date
        # M2提取小时
        df['hour'] = df['datetime'].dt.hour

        # 处理异常值
        df['pct_change'] = pd.to_numeric(df['pct_change'], errors='coerce').fillna(0)

        return df

    except FileNotFoundError:
        print("错误：找不到CSV文件，请确保数据文件在正确路径下。")
        return None
    except Exception as e:
        print(f"发生未知错误：{e}")
        return None


def feature_engineering_1(df):
    """特征1：指数在不同时间段的平均涨跌幅"""
    print("\n开始构建特征1：指数在不同时间段的平均涨跌幅")

    def time_period(hour):
        if 9 <= hour < 11:
            return 'morning'
        elif 11 <= hour < 14:
            return 'noon'
        else:
            return 'afternoon'

    df['time_period'] = df['hour'].apply(time_period)

    # M3 指数×日期×时段 平均涨跌幅
    period_index_change = (
        df.groupby(['index_id', 'date_only', 'time_period'], as_index=False)['pct_change']
        .mean()
        .rename(columns={'pct_change': 'avg_pct_change'})
    )
    print(f"计算了 {len(period_index_change)} 个指数×日期的平均涨跌幅")
    print(f"平均涨跌幅: {period_index_change['avg_pct_change'].mean():.4f}")

    return df, period_index_change


def feature_engineering_2(df):
    """特征2：板块 × 市场阶段的历史涨跌概率"""
    print("\n开始构建特征2：板块 × 市场阶段的历史涨跌概率")

    df['up'] = (df['pct_change'] > 0).astype(int)
    # M4 板块×阶段 平均涨跌幅 + 上涨概率
    sector_phase_probs = (
        df.groupby(['sector', 'market_phase'], as_index=False)
        .agg(avg_pct_change=('pct_change', 'mean'), up_probability=('up', 'mean'))
    )
    print(f"计算了 {len(sector_phase_probs)} 个板块×市场阶段的历史涨跌概率")
    print(f"平均涨跌概率: {sector_phase_probs['up_probability'].mean():.4f}")

    return df, sector_phase_probs


def feature_engineering_3(df):
    """特征3：指数 × 板块的短期涨跌趋势偏好"""
    print("\n开始构建特征3：指数 × 板块的短期涨跌趋势偏好")

    df = df.sort_values(['index_id', 'datetime']).copy()
    df['trend_5'] = (
        df.groupby('index_id')['pct_change']
        .rolling(window=5, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
    )
    # M5 指数×板块 过去5期趋势均值
    index_sector_trend = (
        df.groupby(['index_id', 'sector'], as_index=False)['trend_5']
        .mean()
        .rename(columns={'trend_5': 'avg_trend_5'})
    )
    print(f"计算了 {len(index_sector_trend)} 个指数×板块的短期涨跌趋势")
    print(f"平均趋势值: {index_sector_trend['avg_trend_5'].mean():.4f}")

    return df, index_sector_trend


def build_golden_test_set(df):
    """构建黄金测试集并进行偏差检测"""
    print("\n开始构建黄金测试集并进行偏差检测...")

    df_sorted = df.sort_values('datetime').reset_index(drop=True)

    # M6 时序划分：TimeSeriesSplit 取最后一折；同时给出简单比例切分兜底
    tscv = TimeSeriesSplit(n_splits=5)
    train_idx, test_idx = list(tscv.split(df_sorted))[-1]
    train_data = df_sorted.iloc[train_idx].copy()
    test_data = df_sorted.iloc[test_idx].copy()

    print(f"训练集样本数: {len(train_data)}, 测试集样本数: {len(test_data)}")
    print(f"训练时间范围: {train_data['datetime'].min()} ~ {train_data['datetime'].max()}")
    print(f"测试时间范围: {test_data['datetime'].min()} ~ {test_data['datetime'].max()}")

    # 偏差分析：不同板块涨跌概率分布
    print("\n偏差分析：不同板块涨跌概率")
    if 'up' not in df_sorted.columns:
        df_sorted['up'] = (df_sorted['pct_change'] > 0).astype(int)
        train_data['up'] = (train_data['pct_change'] > 0).astype(int)
        test_data['up'] = (test_data['pct_change'] > 0).astype(int)

    overall_probs = df_sorted.groupby('sector')['up'].mean()
    test_probs = test_data.groupby('sector')['up'].mean()

    all_sectors = sorted(set(df_sorted['sector'].unique()).union(set(test_data['sector'].unique())))
    overall_probs = overall_probs.reindex(all_sectors, fill_value=0)
    test_probs = test_probs.reindex(all_sectors, fill_value=0)
    print(pd.DataFrame({'overall_up_prob': overall_probs, 'test_up_prob': test_probs}))

    # M7 KS检验：测试集 vs 全量 的涨跌幅分布
    ks_stat, ks_p = stats.ks_2samp(test_data['pct_change'], df_sorted['pct_change'])
    print(f"KS检验结果(全量 vs 测试 pct_change): statistic={ks_stat:.4f}, p-value={ks_p:.4f}")

    # 分板块 KS（示例）
    print("\n分板块 KS（测试 vs 全量同板块）:")
    for sec in all_sectors:
        a = test_data.loc[test_data['sector'] == sec, 'pct_change']
        b = df_sorted.loc[df_sorted['sector'] == sec, 'pct_change']
        if len(a) >= 2 and len(b) >= 2:
            s, p = stats.ks_2samp(a, b)
            print(f"  {sec}: statistic={s:.4f}, p-value={p:.4f}")

    return train_data, test_data


def main():
    """主函数，执行完整的数据处理流程"""
    # 原框架写的是 2.1.1.csv；考场文件名为 B2.csv
    df = load_and_preprocess_data('B2.csv')
    if df is None:
        return

    df, period_index_change = feature_engineering_1(df)
    df, sector_phase_probs = feature_engineering_2(df)
    df, index_sector_trend = feature_engineering_3(df)

    high_quality_train = df[['index_id', 'date_only', 'time_period', 'pct_change', 'sector', 'market_phase']]
    high_quality_train.to_csv('high_quality_train_index.csv', index=False)
    print("\n高质量训练集已保存为 high_quality_train_index.csv")

    train_data, test_data = build_golden_test_set(df)
    print("\n所有任务已完成！")


if __name__ == "__main__":
    main()
